[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-3/dynamic-breakpoints.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239526-lesson-4-dynamic-breakpoints)

# 动态断点

## 回顾

我们讨论了人工参与循环的动机：

(1) `批准` - 我们可以中断我们的agent，向用户显示状态，并允许用户接受某个行动

(2) `调试` - 我们可以回退图以重现或避免问题

(3) `编辑` - 你可以修改状态

我们介绍了断点作为在特定步骤停止图的通用方法，这使得像`批准`这样的用例成为可能

我们还展示了如何编辑图状态，并引入人工反馈。

## 目标

断点是由开发者在图编译期间在特定节点上设置的。

但是，有时让图**动态中断**自己是很有帮助的！

这是一个内部断点，[可以使用`NodeInterrupt`来实现](https://langchain-ai.github.io/langgraph/how-tos/human_in_the_loop/dynamic_breakpoints/#run-the-graph-with-dynamic-interrupt)。

这有几个特定的好处：

(1) 你可以有条件地执行它（从节点内部基于开发者定义的逻辑）。

(2) 你可以向用户传达中断的原因（通过向`NodeInterrupt`传递任何你想要的内容）。

让我们创建一个图，其中基于输入的长度抛出`NodeInterrupt`。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph langchain_openai langgraph_sdk

In [ ]:
from IPython.display import Image, display

from typing_extensions import TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langgraph.errors import NodeInterrupt
from langgraph.graph import START, END, StateGraph

class State(TypedDict):
    input: str

def step_1(state: State) -> State:
    """第一步处理函数"""
    print("---第一步---")
    return state

def step_2(state: State) -> State:
    """第二步处理函数，如果输入长度超过5个字符则可选择抛出NodeInterrupt"""
    # 如果输入长度超过5个字符，我们可选择抛出NodeInterrupt
    if len(state['input']) > 5:
        raise NodeInterrupt(f"接收到长度超过5个字符的输入: {state['input']}")
    
    print("---第二步---")
    return state

def step_3(state: State) -> State:
    """第三步处理函数"""
    print("---第三步---")
    return state

builder = StateGraph(State)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)
builder.add_edge(START, "step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

# 设置内存
memory = MemorySaver()

# 使用内存编译图
graph = builder.compile(checkpointer=memory)

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

让我们使用长度超过5个字符的输入来运行图。

In [ ]:
initial_input = {"input": "hello world"}
thread_config = {"configurable": {"thread_id": "1"}}

# 运行图直到第一次中断
for event in graph.stream(initial_input, thread_config, stream_mode="values"):
    print(event)

如果我们此时检查图状态，我们看到下一个要执行的节点（`step_2`）。

In [ ]:
state = graph.get_state(thread_config)
print(state.next)

我们可以看到`Interrupt`被记录到状态中。

In [ ]:
print(state.tasks)

我们可以尝试从断点恢复图。

但是，这只是重新运行相同的节点！

除非状态发生改变，否则我们将被卡在这里。

In [ ]:
for event in graph.stream(None, thread_config, stream_mode="values"):
    print(event)

In [ ]:
state = graph.get_state(thread_config)
print(state.next)

现在，我们可以更新状态。

In [ ]:
graph.update_state(
    thread_config,
    {"input": "hi"},
)

In [ ]:
for event in graph.stream(None, thread_config, stream_mode="values"):
    print(event)

### 与LangGraph API一起使用

**⚠️ 免责声明**

自从拍摄这些视频以来，我们更新了Studio，使其可以在本地运行并在浏览器中打开。这现在是运行Studio的首选方式（而不是像视频中显示的那样使用桌面应用程序）。请参阅[这里](https://langchain-ai.github.io/langgraph/concepts/langgraph_studio/#local-development-server)的本地开发服务器文档和[这里](https://langchain-ai.github.io/langgraph/how-tos/local-studio/#run-the-development-server)的相关说明。要启动本地开发服务器，请在此模块的`/studio`目录中的终端中运行以下命令：

```
langgraph dev
```

你应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到Studio UI：`https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024`。

In [ ]:
if 'google.colab' in str(get_ipython()):
    raise Exception("很抱歉，Google Colab目前不支持LangGraph Studio")

我们通过SDK连接到它。

In [ ]:
from langgraph_sdk import get_client

# 这是本地开发服务器的URL
URL = "http://127.0.0.1:2024"
client = get_client(url=URL)

# 搜索所有托管的图
assistants = await client.assistants.search()

In [ ]:
thread = await client.threads.create()
input_dict = {"input": "hello world"}

async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="dynamic_breakpoints",
    input=input_dict,
    stream_mode="values",):
    
    print(f"接收到类型为: {chunk.event}的新事件...")
    print(chunk.data)
    print("\n\n")

In [ ]:
current_state = await client.threads.get_state(thread['thread_id'])

In [ ]:
current_state['next']

In [ ]:
await client.threads.update_state(thread['thread_id'], {"input": "hi!"})

In [ ]:
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="dynamic_breakpoints",
    input=None,
    stream_mode="values",):
    
    print(f"接收到类型为: {chunk.event}的新事件...")
    print(chunk.data)
    print("\n\n")

In [ ]:
current_state = await client.threads.get_state(thread['thread_id'])
current_state